# 08 — Bipartite Max-Cut: Known-Optimum and Solver-Recovery Analysis

This notebook examines bipartite graph instances whose edge weights are all non-negative. For these graphs, the optimal Max-Cut value is **known analytically**: placing the two vertex partitions on opposite sides of the cut includes every edge, so

\[
C^*(G)=\sum_{(u,v)\in E} w_{uv}.
\]

Because all edge weights are non-negative, no other cut can have a greater weight than the sum of all edge weights. This gives a ground-truth objective against which solver output can be checked, unlike a *best-observed* benchmark reference.

**Reading guide**

1. Load and filter the benchmark instances.
2. Describe the eligible graph population and solver availability.
3. Reconstruct the analytical optimum from the original graph files.
4. Evaluate six solvers on a **common set** of instances with complete results.
5. Inspect instances whose known optimum was not recovered and check numerical tolerances.
6. Repeat key metrics on the Medium/Large sensitivity subset.

**Important denominators.** The original analysis contains **146** eligible graphs (bipartite, zero negative edges, and `limit > 0.25`). Only **136** have all six solver results and are used in the *common-six-solver* comparison. The Medium/Large sensitivity analysis has **66** instances. These populations must not be confused when interpreting recovery percentages.

## 1. Environment and input data

Import the repository helpers and load the preprocessed benchmark dataset. `config.py` defines category order, solver names, and data directories; `utils.py` handles the graph archives. The IPython autoreload magic is retained from the original notebook for interactive development.

In [1]:
# Repository modules are resolved relative to the notebooks/ directory.
# Run from that directory so the relative path points to src/maxcut.

from pathlib import Path
import sys
import numpy as np
import pandas as pd

#project imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils
import plotting

**Input file.** `baseline_dataset.csv` contains graph metadata, solver outcomes, time limits, and the stored best-of-six objective.

In [2]:
# import paths
PATH_BASELINE_DATASET = cfg.PROCESSED_DIR / 'baseline_dataset.csv'

Inspect a few rows to verify that the expected columns and metadata were loaded.

In [3]:
df_baseline = pd.read_csv(PATH_BASELINE_DATASET)
df_baseline.head()

,name,nodes,edges,density,limit,has_leafs,int_only,size_cat,density_cat,heuristics_name,...,neg_wt_sum,neg_wt_ratio,triangle_count,has_triangles,transitivity,average_clustering,cyclomatic_number,bridge_count,bridge_fraction,unique_weights
0,G55,4969,12498,0.001013,13.945,True,True,large,sparse,BURER2002,...,0.0,0.000000,23,True,0.001100,0.001189,7530,180,0.014402,1
1,G56,4969,12498,0.001013,13.922,True,True,large,sparse,BURER2002,...,-6276.0,0.502160,23,True,0.001100,0.001189,7530,180,0.014402,2
2,G57,5000,10000,0.000800,11.299,False,True,large,sparse,BURER2002,...,-5019.0,0.501900,0,False,0.000000,0.000000,5001,0,0.000000,2
3,G58,5000,29570,0.002366,11.876,False,True,large,sparse,BURER2002,...,0.0,0.000000,30784,True,0.098091,0.326822,24571,0,0.000000,1
4,G59,5000,29570,0.002366,15.875,False,True,large,sparse,BURER2002,...,-14737.0,0.498377,30784,True,0.098091,0.326822,24571,0,0.000000,2


**Time-limit inclusion rule.** Reproduce the original filter `limit > 0.25` seconds (strictly greater). This filter is applied *before* structural selection.

In [4]:
# Keep only instances satisfying the study's original time-limit rule.

df_baseline = df_baseline[df_baseline['limit']>0.25]

**Known-optimum eligibility.** `bipartite == True` identifies bipartite graphs, and `neg_edges == 0` excludes negative edge weights. Make a copy to avoid chained-assignment ambiguity in later cells.

In [5]:
# All graph edges can be cut simultaneously for these eligible graphs.
# `.copy()` makes subsequent assignments to df_final explicit and safe.
df_final = df_baseline[
    (df_baseline['bipartite'] == True)
    & (df_baseline['neg_edges'] == 0)
].copy()

print('Eligible bipartite, non-negative-weight instances:', len(df_final))

Eligible bipartite, non-negative-weight instances: 146


## 2. Describe the eligible benchmark population

The following summaries describe the **eligible graph set**, before requiring complete results for all six solvers. Missing solver results affect the denominator of subsequent common-subset analyses.

Count graphs by the repository-defined size categories. `reindex` displays categories in a consistent order, including categories without observations.

In [6]:
# Counts are over df_final, not the later six-solver common subset.

(
    df_final["size_cat"]
    .value_counts()
    .reindex(cfg.SIZE_ORDER)
)

size_cat
xsmall    40
small     30
medium    42
large     24
xlarge    10
Name: count, dtype: int64

List the XLarge instances. Their presence in the eligible population is relevant when interpreting the Medium/Large-only sensitivity subset.

In [7]:
df_final.loc[df_final['size_cat'] == 'xlarge', 'name']

902     g000775
1019    g000901
1037    g000921
1250    g001146
1289    g001188
1946    g001896
2854    g002877
2921    g002948
2940    g002967
3200    g003234
Name: name, dtype: str

Count the eligible graphs by density category. The original results indicate that this known-optimum set belongs to the sparse category.

In [8]:
(
    df_final["density_cat"]
    .value_counts()
    .reindex(cfg.DENSITY_ORDER)
)

density_cat
sparse      146.0
balanced      NaN
dense         NaN
Name: count, dtype: float64

**Availability audit.** A non-null count is shown for each solver column (including any aggregate `heuristic` column in `cfg.SOLVERS`). DAv2 has fewer available results than some other solvers; this is why later comparisons use a complete-case subset.

In [9]:
df_final[cfg.SOLVERS].notna().sum()

DAv2         136
DAv3         146
DAv3c        146
heuristic    146
dtype: int64

Count integer-only and non-integer edge-weight graph instances using the dataset metadata. Non-integer weights are allowed; the essential requirement is **non-negative** weights.

In [10]:
df_final['int_only'].value_counts()

int_only
True     130
False     16
Name: count, dtype: int64

## 3. Compute the analytical optimum from original graphs

The computed optimum is the sum of the original edge weights read from each MQLib graph ZIP file. This is **not** the maximum result reported by the solvers. The graph is located using `utils.find_zip_for_graph()` and parsed using `utils.read_edgelist_from_zip()`.

**Data-integrity detail:** the earlier loop appended sums to a list and then assigned the list to DataFrame rows by position. If a ZIP was missing or its parsing failed, later values could be attached to the wrong graph. The updated loop stores each result by **graph name**, and fails clearly if any eligible graph could not be read. It also avoids reading every ZIP twice. The objective calculation itself, `sum(e[2] for e in edges)`, is unchanged.

In [11]:
# Map graph names to certified Max-Cut objective values.
# Using names rather than list positions prevents silent misalignment.
optimum_by_graph = {}
failed_graphs = []
total = len(df_final)

for i, graph_name in enumerate(df_final['name'], 1):
    progress = (i / total) * 100
    sys.stdout.write(
        f"\rProcessed {i}/{total} ({progress:.2f}%) | Current: {graph_name}"
    )
    sys.stdout.flush()

    zip_path = utils.find_zip_for_graph(graph_name)
    if zip_path is None:
        failed_graphs.append((graph_name, "ZIP archive not found"))
        continue

    try:
        # Each edge record is expected to contain (u, v, weight, ...).
        _, edges, *_ = utils.read_edgelist_from_zip(zip_path)
        optimum_by_graph[graph_name] = sum(e[2] for e in edges)
    except Exception as exc:
        failed_graphs.append((graph_name, str(exc)))

print()  # End the in-place progress display on a new line.
print('Graph optima computed:', len(optimum_by_graph))

if failed_graphs:
    display(pd.DataFrame(failed_graphs, columns=['name', 'error']))
    raise RuntimeError(
        f'Cannot certify all optima: {len(failed_graphs)} graph files failed. '
        'Restore the files and rerun this cell.'
    )

Processed 146/146 (100.00%) | Current: g003251
Graph optima computed: 146


Verify that one optimum was computed for every eligible graph.

In [12]:
# Every graph passing the structural filters must be represented.
assert len(optimum_by_graph) == len(df_final), (
    'Number of computed optima does not match the eligible graph count'
)
len(optimum_by_graph)

146

Attach the graph-derived optimum to its matching dataset row using the name as the key. This preserves the original output-column name, `Objective Value`, required by all subsequent analysis cells.

In [13]:
# Explicit name-based alignment; no dependence on the order of file reads.
df_final['Objective Value'] = df_final['name'].map(optimum_by_graph)
assert df_final['Objective Value'].notna().all(), 'Missing mapped optimum values' 

Inspect the columns to confirm that `Objective Value` is now available alongside solver outputs and metadata.

In [14]:
df_final.columns

Index(['name', 'nodes', 'edges', 'density', 'limit', 'has_leafs', 'int_only',
       'size_cat', 'density_cat', 'heuristics_name', 'DAv2', 'DAv2_runtime',
       'DAv3', 'DAv3_runtime', 'DAv3c', 'DAv3c_runtime', 'BURER2002',
       'BURER2002_runtime', 'MERZ1999GLS', 'MERZ1999GLS_runtime',
       'PAL2004bMTS2', 'PAL2004bMTS2_runtime', 'heuristic',
       'heuristic_runtime', 'baseline_objective', 'best_solvers', 'bipartite',
       'neg_edges', 'neg_edges_frac', 'neg_wt_sum', 'neg_wt_ratio',
       'triangle_count', 'has_triangles', 'transitivity', 'average_clustering',
       'cyclomatic_number', 'bridge_count', 'bridge_fraction',
       'unique_weights', 'Objective Value'],
      dtype='str')

## 4. Six-solver common-instance evaluation

The comparison uses **DAv2, DAv3, DAv3c, BURER2002, MERZ1999GLS, and PAL2004bMTS2**. The original `baseline_objective` is the best observed cut **among these six solver columns**, not a known optimum. All six solver results and both objective columns must be present for an instance to enter `common_df`.

For solver \(s\) and graph \(G\), the metrics are:

- **Relative success:** \(C_s(G) = C_{\mathrm{best6}}(G)\), allowing ties.
- **Exact optimum recovery:** \(C_s(G) = C^*(G)\).
- **Absolute optimality gap:** \(C^*(G)-C_s(G)\).
- **Percentage optimality gap:** \(100[C^*(G)-C_s(G)]/C^*(G)\), when \(C^*(G)>0\).

Unless noted otherwise, numerical equality uses `np.isclose(..., atol=1e-9, rtol=0)`. Both stored-baseline consistency and the condition that no solver exceed the known optimum are checked before interpreting results.

In [15]:
# This is the main complete-case table, which gives comparable denominators
# for the six solvers and validates the stored best-of-six reference.

import numpy as np
import pandas as pd

solvers = [
    "DAv2",
    "DAv3",
    "DAv3c",
    "BURER2002",
    "MERZ1999GLS",
    "PAL2004bMTS2"
]

opt_col = "Objective Value"
baseline_col = "baseline_objective"
EPS = 1e-9

# Use the same instances for all six solvers
required_cols = solvers + [opt_col, baseline_col]

common_df = df_final.dropna(subset=required_cols).copy()

print("Common instances:", len(common_df))

# Verify the stored six-solver baseline
common_df["best_of_six_check"] = common_df[solvers].max(axis=1)

baseline_matches = np.isclose(
    common_df[baseline_col],
    common_df["best_of_six_check"],
    atol=EPS,
    rtol=0
)

print("Baseline mismatches:", (~baseline_matches).sum())

# Verify that no solver exceeds the certified optimum
above_optimum = (
    common_df[solvers].gt(
        common_df[opt_col] + EPS,
        axis=0
    )
)

print("Solver values exceeding optimum:", above_optimum.sum().sum())

assert baseline_matches.all(), "Stored baseline differs from best of six"
assert not above_optimum.any().any(), "Some solver values exceed the optimum"

rows = []

for solver in solvers:

    values = common_df[solver].to_numpy(float)
    optimum = common_df[opt_col].to_numpy(float)
    baseline = common_df[baseline_col].to_numpy(float)

    # Relative success: best among six
    relative_success = np.isclose(
        values, baseline,
        atol=EPS,
        rtol=0
    )

    # Exact recovery: matches known optimum
    exact_recovery = np.isclose(
        values, optimum,
        atol=EPS,
        rtol=0
    )

    # Absolute optimality gap in objective units
    absolute_gap = optimum - values

    # Relative optimality gap (%), defined when optimum > 0
    relative_gap = np.where(
        optimum > 0,
        100 * absolute_gap / np.where(optimum > 0, optimum, 1),
        np.nan
    )

    rows.append({
        "solver": solver,
        "N": len(common_df),

        "relative_success_n": int(relative_success.sum()),
        "relative_success_pct": 100 * relative_success.mean(),

        "exact_recovery_n": int(exact_recovery.sum()),
        "exact_recovery_pct": 100 * exact_recovery.mean(),

        "median_optimality_gap_pct": np.nanmedian(relative_gap),
        "mean_optimality_gap_pct": np.nanmean(relative_gap),

        "max_optimality_gap_pct": np.nanmax(relative_gap)
    })

opt_df = pd.DataFrame(rows)

display(opt_df.round(4))

Common instances: 136
Baseline mismatches: 0
Solver values exceeding optimum: 0


,solver,N,relative_success_n,relative_success_pct,exact_recovery_n,exact_recovery_pct,median_optimality_gap_pct,mean_optimality_gap_pct,max_optimality_gap_pct
0,DAv2,136,125,91.9118,125,91.9118,0.0000,0.0237,0.8159
1,DAv3,136,133,97.7941,131,96.3235,0.0000,0.0228,2.3651
2,DAv3c,136,101,74.2647,100,73.5294,0.0000,0.1961,1.9374
3,BURER2002,136,101,74.2647,101,74.2647,0.0000,0.2080,1.4081
4,MERZ1999GLS,136,49,36.0294,49,36.0294,0.9628,1.0361,2.6636
5,PAL2004bMTS2,136,56,41.1765,55,40.4412,0.8096,0.8599,3.3800


### 4.1 Does *any* evaluated solver recover the true optimum?

Compare the best of the six observed cuts with the analytically known optimum. List any graphs where all six miss, because a best-observed solution should not be mistaken for an optimum.

In [16]:
# Each row is a graph; the best-of-six reaches the true optimum
# only if at least one of these six solver runs is optimal.

# Does at least one of the six solvers recover the true optimum?
best_six_optimal = np.isclose(
    common_df["best_of_six_check"],
    common_df[opt_col],
    atol=EPS,
    rtol=0
)

print("Total instances:", len(common_df))
print("Best-of-six recovers optimum:", best_six_optimal.sum())
print("Best-of-six recovery rate (%):", 100 * best_six_optimal.mean())

print(
    "Instances where all six solvers miss the optimum:",
    (~best_six_optimal).sum()
)

display(
    common_df.loc[
        ~best_six_optimal,
        ["name", opt_col, baseline_col] + solvers
    ]
)

Total instances: 136
Best-of-six recovers optimum: 132
Best-of-six recovery rate (%): 97.05882352941177
Instances where all six solvers miss the optimum: 4


,name,Objective Value,baseline_objective,DAv2,DAv3,DAv3c,BURER2002,MERZ1999GLS,PAL2004bMTS2
1103,g000989,2322.000000,2319.000000,2318.000000,2319.000000,2300.000000,2299.000000,2283.000000,2288.000000
2009,g001966,1465.000000,1464.000000,1462.000000,1464.000000,1453.000000,1453.000000,1436.000000,1440.000000
2175,g002147,61065.170469,61055.337667,60566.921113,61019.943519,60984.720606,60991.783050,61040.520753,61055.337667
2210,g002183,51109.374690,51109.374690,50791.779036,50874.118650,51109.374690,50438.894976,50274.215748,50344.792560


### 4.2 Gap severity conditional on relative success

Separate each solver’s runs into **relative successes** (best-of-six ties) and **relative failures**. Compute exact-optimum recovery and the distribution of percentage gaps within each group. This helps distinguish *winning the comparison* from *solving the graph optimally*.

In [17]:
# A relative winner can still miss the analytical optimum
# if every evaluated solver misses it.

quality_rows = []

for solver in solvers:

    values = common_df[solver].to_numpy(float)
    optimum = common_df[opt_col].to_numpy(float)
    baseline = common_df[baseline_col].to_numpy(float)

    relative_success = np.isclose(
        values, baseline,
        atol=EPS,
        rtol=0
    )

    exact_recovery = np.isclose(
        values, optimum,
        atol=EPS,
        rtol=0
    )

    gap_pct = np.where(
        optimum > 0,
        100 * (optimum - values) / np.where(optimum > 0, optimum, 1),
        np.nan
    )

    for status, mask in [
        ("Relative success", relative_success),
        ("Relative failure", ~relative_success)
    ]:
        selected_gaps = gap_pct[mask]

        quality_rows.append({
            "solver": solver,
            "relative_status": status,
            "N": int(mask.sum()),
            "exact_recovery_n": int(exact_recovery[mask].sum()),
            "exact_recovery_pct": (
                100 * exact_recovery[mask].mean()
                if mask.sum() > 0 else np.nan
            ),
            "median_gap_pct": (
                np.nanmedian(selected_gaps)
                if mask.sum() > 0 else np.nan
            ),
            "mean_gap_pct": (
                np.nanmean(selected_gaps)
                if mask.sum() > 0 else np.nan
            ),
            "p90_gap_pct": (
                np.nanpercentile(selected_gaps, 90)
                if mask.sum() > 0 else np.nan
            )
        })

quality_df = pd.DataFrame(quality_rows)

display(quality_df.round(4))

,solver,relative_status,N,exact_recovery_n,exact_recovery_pct,median_gap_pct,mean_gap_pct,p90_gap_pct
0,DAv2,Relative success,125,125,100.0000,0.0000,0.0000,0.0000
1,DAv2,Relative failure,11,0,0.0000,0.1926,0.2932,0.6214
2,DAv3,Relative success,133,131,98.4962,0.0000,0.0015,0.0000
3,DAv3,Relative failure,3,0,0.0000,0.4603,0.9665,1.9842
4,DAv3c,Relative success,101,100,99.0099,0.0000,0.0000,0.0000
5,DAv3c,Relative failure,35,0,0.0000,0.8191,0.7621,1.2072
6,BURER2002,Relative success,101,101,100.0000,0.0000,0.0000,0.0000
7,BURER2002,Relative failure,35,0,0.0000,0.8445,0.8081,1.2690
8,MERZ1999GLS,Relative success,49,49,100.0000,0.0000,0.0000,0.0000
9,MERZ1999GLS,Relative failure,87,0,0.0000,1.6174,1.6196,2.4589


## 5. Diagnostic inspection of unrecovered optima

The next cells intentionally retain detailed checks from the original exploratory analysis. Some repeat the best-of-six recovery count, but provide different views into missed instances, objective gaps, and comparison tolerance. These checks support auditability; they do not define a new benchmark population.

Recreate the subset where the stored six-solver baseline fails to reach the known optimum. The earlier cell used the recomputed best-of-six value, so this is a cross-check against the stored reference.

In [18]:
# Cross-check the missed instances using the stored baseline_objective.

EPS = 1e-9

common_df["best_of_six_optimal"] = np.isclose(
    common_df["baseline_objective"],
    common_df["Objective Value"],
    atol=EPS,
    rtol=0
)

missed_df = common_df.loc[
    ~common_df["best_of_six_optimal"]
].copy()

print("Total instances:", len(common_df))
print(
    "At least one solver recovered the optimum:",
    common_df["best_of_six_optimal"].sum()
)
print(
    "No solver recovered the optimum:",
    len(missed_df)
)

display(
    missed_df[
        ["name", "Objective Value", "baseline_objective"]
        + solvers
    ]
)

Total instances: 136
At least one solver recovered the optimum: 132
No solver recovered the optimum: 4


,name,Objective Value,baseline_objective,DAv2,DAv3,DAv3c,BURER2002,MERZ1999GLS,PAL2004bMTS2
1103,g000989,2322.000000,2319.000000,2318.000000,2319.000000,2300.000000,2299.000000,2283.000000,2288.000000
2009,g001966,1465.000000,1464.000000,1462.000000,1464.000000,1453.000000,1453.000000,1436.000000,1440.000000
2175,g002147,61065.170469,61055.337667,60566.921113,61019.943519,60984.720606,60991.783050,61040.520753,61055.337667
2210,g002183,51109.374690,51109.374690,50791.779036,50874.118650,51109.374690,50438.894976,50274.215748,50344.792560


Inspect absolute and percentage gaps at high display precision. The `isclose_rel_1e10` column applies a **scale-aware tolerance for diagnosis only**; it is not the main exact-recovery rule and should not be conflated with it.

In [19]:
# Diagnostic comparison: absolute-only tolerance vs a relaxed relative tolerance.
# A nonzero relative tolerance may treat tiny but nonzero differences as ties.

import numpy as np
import pandas as pd

EPS = 1e-9

missed_check = missed_df.copy()

missed_check["absolute_gap"] = (
    missed_check["Objective Value"]
    - missed_check["baseline_objective"]
)

missed_check["relative_gap_pct"] = (
    100 * missed_check["absolute_gap"]
    / missed_check["Objective Value"]
)

missed_check["isclose_abs_1e9"] = np.isclose(
    missed_check["Objective Value"],
    missed_check["baseline_objective"],
    atol=1e-9,
    rtol=0
)

# Diagnostic only: a scale-aware tolerance
missed_check["isclose_rel_1e10"] = np.isclose(
    missed_check["Objective Value"],
    missed_check["baseline_objective"],
    atol=1e-9,
    rtol=1e-10
)

with pd.option_context("display.precision", 15):
    display(
        missed_check[
            [
                "name",
                "Objective Value",
                "baseline_objective",
                "absolute_gap",
                "relative_gap_pct",
                "isclose_abs_1e9",
                "isclose_rel_1e10"
            ]
        ]
    )

,name,Objective Value,baseline_objective,absolute_gap,relative_gap_pct,isclose_abs_1e9,isclose_rel_1e10
1103,g000989,2322.000000000000000,2319.000000000000000,3.000000000000000,0.129198966408269,False,False
2009,g001966,1465.000000000000000,1464.000000000000000,1.000000000000000,0.068259385665529,False,False
2175,g002147,61065.170469061384210,61055.337666508588882,9.832802552795329,0.016102145424743,False,False
2210,g002183,51109.374690000004193,51109.374689997173846,0.000000002830348,0.000000000005538,False,True


For each graph missed by every solver, identify which of the six solvers obtained the **best observed** cut. There can be multiple relative winners because ties are allowed.

In [20]:
rows = []

for _, row in missed_df.iterrows():

    best = row["baseline_objective"]
    optimum = row["Objective Value"]

    relative_winners = [
        s for s in solvers
        if np.isclose(row[s], best, atol=EPS, rtol=0)
    ]

    rows.append({
        "name": row["name"],
        "absolute_gap": optimum - best,
        "relative_winners": ", ".join(relative_winners)
    })

display(pd.DataFrame(rows))

,name,absolute_gap,relative_winners
0,g000989,3.000000e+00,DAv3
1,g001966,1.000000e+00,DAv3
2,g002147,9.832803e+00,PAL2004bMTS2
3,g002183,2.830348e-09,DAv3c


### 5.1 Alternative tolerance sensitivity

The original notebook additionally computed recovery with `atol=1e-9, rtol=1e-10`. **Treat this as a tolerance-sensitivity analysis**, not as the primary strict count. Any differences between this table and the strict `rtol=0` results must be reported as consequences of the tolerance definition.

In [21]:
# Sensitivity check only; primary exact-recovery rates use rtol=0.

ATOL = 1e-9
RTOL = 1e-10

rows = []

for solver in solvers:
    recovered = np.isclose(
        common_df[solver].to_numpy(float),
        common_df["Objective Value"].to_numpy(float),
        atol=ATOL,
        rtol=RTOL
    )

    rows.append({
        "solver": solver,
        "N": len(common_df),
        "exact_recovery_n": int(recovered.sum()),
        "exact_recovery_pct": 100 * recovered.mean()
    })

recovery_df = pd.DataFrame(rows)
display(recovery_df.round(4))

,solver,N,exact_recovery_n,exact_recovery_pct
0,DAv2,136,125,91.9118
1,DAv3,136,131,96.3235
2,DAv3c,136,101,74.2647
3,BURER2002,136,101,74.2647
4,MERZ1999GLS,136,49,36.0294
5,PAL2004bMTS2,136,55,40.4412


### 5.2 Locate failed recoveries by size

For each solver separately, count graphs in the eligible set where its observed objective differs from the known optimum, grouped by `size_cat`. This cell uses **solver-specific available rows** from `df_final`, not the six-solver common-case set, so its denominators are different.

In [22]:
# Solver-specific availability is intentionally permitted in this diagnostic.
# Do not compare these failure counts directly to common_df rates.

for solver in solvers:

    valid = df_final[[solver, "Objective Value", "size_cat"]].dropna()

    failed = valid[
    ~np.isclose(
        valid[solver],
        valid["Objective Value"],
        atol=1e-9,
        rtol=0
    )
]

    print(f"\n===== {solver} failures by size =====")

    print(
        failed["size_cat"]
        .value_counts()
        .reindex(cfg.SIZE_ORDER)
    )


===== DAv2 failures by size =====
size_cat
xsmall    NaN
small     1.0
medium    6.0
large     4.0
xlarge    NaN
Name: count, dtype: float64

===== DAv3 failures by size =====
size_cat
xsmall     1
small      1
medium     2
large      1
xlarge    10
Name: count, dtype: int64

===== DAv3c failures by size =====
size_cat
xsmall     1
small      4
medium    16
large     15
xlarge     6
Name: count, dtype: int64

===== BURER2002 failures by size =====
size_cat
xsmall     NaN
small      1.0
medium    18.0
large     16.0
xlarge    10.0
Name: count, dtype: float64

===== MERZ1999GLS failures by size =====
size_cat
xsmall    20
small     18
medium    31
large     18
xlarge    10
Name: count, dtype: int64

===== PAL2004bMTS2 failures by size =====
size_cat
xsmall    22
small     18
medium    24
large     17
xlarge    10
Name: count, dtype: int64


## 6. Medium/Large sensitivity analysis

To checks whether conclusions change after restricting the common comparison to `medium` and `large` instances, the original notebook expects **66** such graphs (42 Medium and 24 Large). This is a subset of the 136 common complete-case graphs, **not** a fresh graph selection independent of the full analysis.

The function below recomputes the relative-success, exact-recovery, and gap metrics for a supplied subset, verifying the best-of-six baseline. It assumes strictly positive optima because it directly divides by `Objective Value` to compute percentage gaps.

Select and validate the 66-instance Medium/Large subset. The explicit count assertion is a deliberate reproducibility guard: it highlights dataset changes rather than silently altering the paper’s reported denominator.

In [23]:
# Select medium and large instances
df_66 = common_df.loc[
    common_df["size_cat"].isin(["medium", "large"])
].copy()

print("Total instances:", len(df_66))
print("\nSize distribution:")
print(df_66["size_cat"].value_counts())

assert len(df_66) == 66, (
    f"Expected 66 instances, found {len(df_66)}"
)

assert df_66[solvers + ["Objective Value", "baseline_objective"]].notna().all().all()

Total instances: 66

Size distribution:
size_cat
medium    42
large     24
Name: count, dtype: int64


Apply the same evaluation function to all **136** common instances and then to the **66** Medium/Large instances. The variable names are retained for compatibility with the original conference-paper analysis.

In [24]:
# For an apples-to-apples sensitivity comparison, both evaluations use
# the same six solver columns, equality rule, and baseline definition.

import numpy as np
import pandas as pd

EPS = 1e-9

def evaluate_known_optima(df, solvers, eps=1e-9):
    rows = []

    # Recompute and verify the best of six
    best_of_six = df[solvers].max(axis=1).to_numpy(float)
    optimum = df["Objective Value"].to_numpy(float)

    stored_baseline = df["baseline_objective"].to_numpy(float)

    assert np.isclose(
        best_of_six, stored_baseline,
        atol=eps, rtol=0
    ).all(), "Mismatch in stored six-solver baseline"

    assert np.all(optimum > 0), (
        "Relative gap requires positive optimum values"
    )

    assert np.all(best_of_six <= optimum + eps), (
        "Best-of-six objective exceeds the known optimum"
    )

    for solver in solvers:
        values = df[solver].to_numpy(float)

        relative_success = np.isclose(
            values, best_of_six,
            atol=eps, rtol=0
        )

        exact_recovery = np.isclose(
            values, optimum,
            atol=eps, rtol=0
        )

        gap_pct = 100 * (optimum - values) / optimum

        rows.append({
            "Solver": solver,
            "N": len(df),
            "Relative_success_n": int(relative_success.sum()),
            "Relative_success_%": 100 * relative_success.mean(),
            "Exact_recovery_n": int(exact_recovery.sum()),
            "Exact_recovery_%": 100 * exact_recovery.mean(),
            "Mean_gap_%": gap_pct.mean(),
            "Median_gap_%": np.median(gap_pct),
            "Max_gap_%": gap_pct.max()
        })

    return pd.DataFrame(rows)

results_136 = evaluate_known_optima(common_df, solvers)
results_66 = evaluate_known_optima(df_66, solvers)

print("=== FULL KNOWN-OPTIMUM SUBSET (N=136) ===")
display(results_136.round(4))

print("=== MEDIUM/LARGE SENSITIVITY SUBSET (N=66) ===")
display(results_66.round(4))

=== FULL KNOWN-OPTIMUM SUBSET (N=136) ===


,Solver,N,Relative_success_n,Relative_success_%,Exact_recovery_n,Exact_recovery_%,Mean_gap_%,Median_gap_%,Max_gap_%
0,DAv2,136,125,91.9118,125,91.9118,0.0237,0.0000,0.8159
1,DAv3,136,133,97.7941,131,96.3235,0.0228,0.0000,2.3651
2,DAv3c,136,101,74.2647,100,73.5294,0.1961,0.0000,1.9374
3,BURER2002,136,101,74.2647,101,74.2647,0.2080,0.0000,1.4081
4,MERZ1999GLS,136,49,36.0294,49,36.0294,1.0361,0.9628,2.6636
5,PAL2004bMTS2,136,56,41.1765,55,40.4412,0.8599,0.8096,3.3800


=== MEDIUM/LARGE SENSITIVITY SUBSET (N=66) ===


,Solver,N,Relative_success_n,Relative_success_%,Exact_recovery_n,Exact_recovery_%,Mean_gap_%,Median_gap_%,Max_gap_%
0,DAv2,66,56,84.8485,56,84.8485,0.0458,0.0000,0.8159
1,DAv3,66,64,96.9697,63,95.4545,0.0101,0.0000,0.4603
2,DAv3c,66,36,54.5455,35,53.0303,0.3381,0.0000,1.5420
3,BURER2002,66,32,48.4848,32,48.4848,0.4161,0.1225,1.4081
4,MERZ1999GLS,66,17,25.7576,17,25.7576,1.3771,1.5557,2.6636
5,PAL2004bMTS2,66,26,39.3939,25,37.8788,1.0136,0.9442,2.6534


Compare recovery rates, relative-success rates, and mean gaps between the two populations. Percentage-point changes are computed as **Medium/Large minus full common subset**, not as relative percent changes.

In [25]:
# A negative recovery change means a lower exact-optimum hit rate
# on the Medium/Large subset than in the full common subset.

comparison = results_136.merge(
    results_66,
    on="Solver",
    suffixes=("_136", "_66"),
    validate="one_to_one"
)

comparison["Recovery_change_pp"] = (
    comparison["Exact_recovery_%_66"]
    - comparison["Exact_recovery_%_136"]
)

comparison["Relative_success_change_pp"] = (
    comparison["Relative_success_%_66"]
    - comparison["Relative_success_%_136"]
)

display(
    comparison[
        [
            "Solver",
            "Relative_success_%_136",
            "Relative_success_%_66",
            "Exact_recovery_%_136",
            "Exact_recovery_%_66",
            "Recovery_change_pp",
            "Mean_gap_%_136",
            "Mean_gap_%_66"
        ]
    ].round(3)
)

,Solver,Relative_success_%_136,Relative_success_%_66,Exact_recovery_%_136,Exact_recovery_%_66,Recovery_change_pp,Mean_gap_%_136,Mean_gap_%_66
0,DAv2,91.912,84.848,91.912,84.848,-7.063,0.024,0.046
1,DAv3,97.794,96.970,96.324,95.455,-0.869,0.023,0.010
2,DAv3c,74.265,54.545,73.529,53.030,-20.499,0.196,0.338
3,BURER2002,74.265,48.485,74.265,48.485,-25.780,0.208,0.416
4,MERZ1999GLS,36.029,25.758,36.029,25.758,-10.272,1.036,1.377
5,PAL2004bMTS2,41.176,39.394,40.441,37.879,-2.562,0.860,1.014


Check which graphs remain unrecovered by the **best of six** in each population. This complements the per-solver tables and keeps the distinction between best-observed and known-optimal objectives explicit.

In [26]:
for label, df in [
    ("All common instances", common_df),
    ("Medium/Large only", df_66)
]:
    optimum = df["Objective Value"].to_numpy(float)
    best_of_six = df[solvers].max(axis=1).to_numpy(float)

    recovered = np.isclose(
        best_of_six,
        optimum,
        atol=EPS,
        rtol=0
    )

    print(f"\n{label}")
    print("N:", len(df))
    print("Best-of-six recovered:", int(recovered.sum()))
    print("Recovery rate (%):", round(100 * recovered.mean(), 2))

    print("Instances not recovered:")
    display(
        df.loc[
            ~recovered,
            ["name", "size_cat", "Objective Value",
             "baseline_objective"]
        ]
    )


All common instances
N: 136
Best-of-six recovered: 132
Recovery rate (%): 97.06
Instances not recovered:


,name,size_cat,Objective Value,baseline_objective
1103,g000989,medium,2322.000000,2319.000000
2009,g001966,small,1465.000000,1464.000000
2175,g002147,large,61065.170469,61055.337667
2210,g002183,medium,51109.374690,51109.374690



Medium/Large only
N: 66
Best-of-six recovered: 63
Recovery rate (%): 95.45
Instances not recovered:


,name,size_cat,Objective Value,baseline_objective
1103,g000989,medium,2322.000000,2319.000000
2175,g002147,large,61065.170469,61055.337667
2210,g002183,medium,51109.374690,51109.374690


Audit the original time-limit distribution of the Medium/Large subset. This is a descriptive check, not a second filtering step; the main inclusion rule was `limit > 0.25`.

In [27]:
print("Minimum time limit:", df_66["limit"].min())

print(
    "Instances with original time limit below 1 second:",
    int((df_66["limit"] < 1).sum())
)

Minimum time limit: 2.609
Instances with original time limit below 1 second: 0
